In [12]:
## ============================================================
##  Marker Gene Visualization Script
##  Input  : /data/work/0marker/marker.txt  (Gene_ID / Name / Summarized_Info)
##           /data/work/your_scRNA.rds
##  Output : /data/work/1results/
## ============================================================

library(Seurat)
library(ggplot2)
library(cowplot)
library(Matrix)
library(dplyr)
library(patchwork)

# ── 0. 参数设置（只需修改这里）────────────────────────────────
prefix      <- "bent_test"  # 🥰 修改样本名称
rds_path    <- "/data/work/RNA_AraSeed/CellTypeAnno/bent/AAsubsetBatch1/scRNA_subset_batch1.rds"
marker_file <- "/data/work/00markers/rysummary.txt"
output_dir  <- "/data/work/RNA_AraSeed/CellTypeAnno/bent/AAsubsetBatch1/Anno"

dir.create(output_dir, showWarnings = FALSE, recursive = TRUE)
setwd(output_dir)

# ── 1. Load data ─────────────────────────────────────────────
scRNA <- readRDS(rds_path)

# 检查marker文件格式
readLines(marker_file, n = 10)
markerinfo <- read.table(marker_file, header = TRUE, sep = "\t", 
                         quote = "", stringsAsFactors = FALSE)

# Expect columns: Gene_ID | Name | Summarized_Info
colnames(markerinfo)[1:3] <- c("Gene_ID", "Name", "Summarized_Info")

# ── 2. Clean & harmonise gene IDs ────────────────────────────
markerinfo <- markerinfo %>%
  mutate(across(everything(), trimws))

all_genes      <- rownames(scRNA)
genes_upper    <- toupper(all_genes)
marker_upper   <- toupper(markerinfo$Gene_ID)

geneList <- all_genes[genes_upper %in% marker_upper]

not_found <- markerinfo$Gene_ID[!(toupper(markerinfo$Gene_ID) %in% genes_upper)]

if (length(not_found) > 0) {
  message("The following ", length(not_found), " marker gene(s) were NOT found in the Seurat object:")
  print(not_found)
} else {
  message("All marker genes matched successfully.")
}
message(length(geneList), " marker genes found in scRNA object out of ",
        nrow(markerinfo), " markers supplied.")
if (length(geneList) == 0) stop("No marker genes matched the scRNA object.")

# ── 3. Build plot_gene labels ────────────────────────────────
markerinfo <- markerinfo %>%
  mutate(
    plot_gene = paste(Gene_ID, Name, Summarized_Info, sep = "|"),
    plot_gene = substr(plot_gene, 1, 50)
  )

marker_map <- setNames(markerinfo$plot_gene, toupper(markerinfo$Gene_ID))

# ── 4. Build full gene-rename table ──────────────────────────
ids <- data.frame(
  Gene     = all_genes,
  plot_gene = all_genes,
  stringsAsFactors = FALSE
)

hit <- match(genes_upper, names(marker_map))
ids$plot_gene[!is.na(hit)] <- marker_map[hit[!is.na(hit)]]

# ── 5. RenameGenesSeurat (修正版) ────────────────────────────
RenameGenesSeurat_v2 <- function(obj, newnames, gene.use = NULL, de.assay = "RNA") {
  lassays   <- Assays(obj)
  assay.use <- if (length(obj@reductions) > 0 && !is.null(obj@reductions$pca)) {
    obj@reductions$pca@assay.used
  } else {
    de.assay
  }
  DefaultAssay(obj) <- de.assay

  if (is.null(gene.use)) {
    all_genenames <- rownames(obj)
  } else {
    all_genenames <- gene.use
    obj <- subset(obj, features = gene.use)
  }

  order_name <- function(v1, v2, ref) {
    df <- data.frame(v1, v2, stringsAsFactors = FALSE)
    rownames(df) <- df$v1
    df[ref, ]
  }

  df1           <- order_name(all_genenames, newnames, rownames(obj))
  all_genenames <- df1$v1
  newnames      <- df1$v2

  change_assay <- function(obj, a1, newnames, all_genenames) {
    RNA <- obj@assays[[a1]]
    if (nrow(RNA) != length(newnames))
      stop("Unequal gene sets: nrow(RNA) != length(newnames)")

    if (length(RNA@counts)) RNA@counts@Dimnames[[1]]    <- newnames
    if (length(RNA@data))   RNA@data@Dimnames[[1]]      <- newnames

    if (length(RNA@var.features)) {
      d <- order_name(all_genenames, newnames, RNA@var.features)
      RNA@var.features <- d$v2
    }
    if (length(RNA@scale.data)) {
      d <- order_name(all_genenames, newnames, rownames(RNA@scale.data))
      rownames(RNA@scale.data) <- d$v2
    }
    obj@assays[[a1]] <- RNA
    obj
  }

  for (a in lassays) {
    DefaultAssay(obj) <- a
    d             <- order_name(all_genenames, newnames, rownames(obj))
    obj           <- change_assay(obj, a, d$v2, d$v1)
  }

  if (length(obj@reductions) > 0 && !is.null(obj@reductions$pca)) {
    hvg <- VariableFeatures(obj, assay = assay.use)
    if (length(hvg) > 0) {
      d <- order_name(all_genenames, newnames, hvg)
      rownames(obj@reductions$pca@feature.loadings) <- d$v2
    }
  }

  obj
}

# ── 6. Prepare Seurat object for plotting ─────────────────────
group_col <- if ("clusters" %in% colnames(scRNA@meta.data)) {
  "clusters"
} else if ("seurat_clusters" %in% colnames(scRNA@meta.data)) {
  "seurat_clusters"
} else {
  message("Available meta.data columns: ",
          paste(colnames(scRNA@meta.data), collapse = ", "))
  stop("Cannot find a cluster column. Set group_col manually to one of the above.")
}
message("Using '", group_col, "' as grouping variable.")
Idents(scRNA) <- scRNA@meta.data[[group_col]]

ids <- ids[match(rownames(scRNA), ids$Gene), ]
stopifnot(!any(is.na(ids$plot_gene)))
stopifnot(nrow(scRNA) == nrow(ids))

scRNA_plot <- RenameGenesSeurat_v2(obj = scRNA, newnames = ids$plot_gene)

my_features <- unique(markerinfo$plot_gene[
  toupper(markerinfo$Gene_ID) %in% genes_upper[genes_upper %in% marker_upper]
])
my_features <- my_features[my_features %in% rownames(scRNA_plot)]
message(length(my_features), " features will be plotted.")

# ── 7. Dynamic figure dimensions ──────────────────────────────
n_feat        <- length(my_features)
n_clusters    <- length(levels(scRNA_plot))
h_dot         <- max(4, ceiling(n_feat / 5) * 4 * 0.4)
w_dot         <- max(6, n_clusters * 0.7)
ncol_feat     <- min(5, n_feat)
nrow_feat     <- ceiling(n_feat / ncol_feat)
w_vln         <- ncol_feat * 4
h_vln         <- nrow_feat * 3
w_feat        <- ncol_feat * 5.2
h_feat        <- nrow_feat * 4

# ── 8. DotPlot ────────────────────────────────────────────────
pdf(file.path(output_dir, paste0(prefix, "_marker_DotPlot.pdf")),
    width = w_dot, height = h_dot)
print(
  DotPlot(scRNA_plot, assay = "RNA", features = my_features) +
    coord_flip() +
    theme(axis.title.x = element_blank(),
          axis.title.y = element_blank()) +
    ggtitle(paste0(prefix, " Marker DotPlot"))
)
dev.off()
message("DotPlot saved: ", file.path(output_dir, paste0(prefix, "_marker_DotPlot.pdf")))

# ── 9. VlnPlot (no points) ───────────────────────────────────
pdf(file.path(output_dir, paste0(prefix, "_marker_VlnPlot.pdf")),
    width = w_vln, height = h_vln)
print(
  VlnPlot(scRNA_plot, features = my_features,
          pt.size = 0, group.by = group_col) + NoLegend()
)
dev.off()
message("VlnPlot saved: ", file.path(output_dir, paste0(prefix, "_marker_VlnPlot.pdf")))

# # # ── 10. FeaturePlot ───────────────────────────────────────────
# pdf(file.path(output_dir, paste0(prefix, "_marker_FeaturePlot.pdf")),
#     width = w_feat, height = h_feat)
# print(
#   FeaturePlot(scRNA_plot, features = my_features,
#               label = TRUE, label.size = 4,
#               order = TRUE, combine = TRUE,
#               ncol = ncol_feat, min.cutoff = "q9")
# )
# dev.off()
# message("FeaturePlot saved: ", file.path(output_dir, paste0(prefix, "_marker_FeaturePlot.pdf")))

# cat("\n✓ All marker plots saved to:", output_dir, "\n")
# # #── 11. 输出每簇细胞数统计 ───────────────────────────────────
# cluster_summary <- as.data.frame(table(scRNA@meta.data$seurat_clusters))
# colnames(cluster_summary) <- c("Cluster", "Cell_Count")

# # 保存为txt文件
# write.table(cluster_summary, 
#             file = file.path(output_dir, paste0(prefix, "_cluster_cell_counts.txt")),
#             sep = "\t", 
#             row.names = FALSE, 
#             quote = FALSE)

[1] "Gene_ID\tName\tSummarized_Info" "AT1G65330\tPHE1\tCZE"          
 [3] "AT1G65300\tPHE2\tCZE"           "AT1G49770\tZOU\tESR"           
 [5] "AT1G62340\tALE1\tSER"           "AT1G55600\tMINI3\tMCE"         
 [7] "AT5G58080\tRR22\tCZSC"          "AT1G53500\tMUM4\toi"           
 [9] "AT1G45231\tTBA1\toi"            "AT1G50010\tTBA2\toi"

All marker genes matched successfully.

16 marker genes found in scRNA object out of 16 markers supplied.

Using 'seurat_clusters' as grouping variable.

16 features will be plotted.



pdf 
  2

DotPlot saved: /data/work/RNA_AraSeed/CellTypeAnno/bent/AAsubsetBatch1/Anno/bent_test_marker_DotPlot.pdf



pdf 
  2

VlnPlot saved: /data/work/RNA_AraSeed/CellTypeAnno/bent/AAsubsetBatch1/Anno/bent_test_marker_VlnPlot.pdf

